# NB02 — Association Analysis
**Pipeline:** Generalised Metabolite–Metagenomics Correlation Study  
**Inputs:** `preprocessed_data.pkl`  
**Outputs:** `analysis_results.pkl`, CSV tables, figures

| Step | Analysis |
|---|---|
| 1–3 | Load data, feature selection (top-variance) |
| 4 | Differential abundance — 3 comparisons (Healthy vs Early, Healthy vs Advanced, Early vs Advanced) |
| 5 | Volcano plots — metabolites + species DA heatmap |
| 6 | Global Spearman correlation matrix (vectorised) |
| 7 | Partial correlations (confounder-adjusted OLS residuals) |
| 8 | Stage-stratified Spearman correlations |
| 9 | Species co-abundance matrix |
| 10 | Hub species / metabolite identification |
| 11 | Kim_adenomas_2020 replication |
| 12 | Save results |


## 1 · Imports

In [2]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path(".").resolve()))

from utils import (
    RESULTS_DIR, INTER_DIR, FIG_DIR, TABLE_DIR,
    DATASET_PRIMARY, DATASET_SECONDARY,
    CRC_STAGE_ORDER, THREE_GROUP_ORDER,
    FDR_THRESHOLD, MIN_CORR,
    MAX_SPECIES_NB02, MAX_MTB_NB02,
    PALETTE_STAGE6, PALETTE_3GROUP,
    load_pickle, save_pickle, savefig,
    differential_abundance,
    spearman_correlation_matrix,
    partial_corr_residuals,
    species_coabundance_matrix,
    annotate_pathway, pathway_kegg_ids,
)
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
%matplotlib inline
%config InlineBackend.figure_format = "retina"
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

for d in [FIG_DIR/"differential", FIG_DIR/"correlation", TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"  Feature caps: {MAX_SPECIES_NB02} species  |  {MAX_MTB_NB02} metabolites")


Imports OK
  Feature caps: 500 species  |  150 metabolites


In [3]:
# ── Analysis Parameters ───────────────────────────────────────────────────────
# Canonical parameter reference for NB02 (Association Analysis).
# All values here are the authoritative source for thesis Methods citations.
PARAMS = dict(
    # Feature selection for correlation analysis
    max_species         = 500,    # top-variance species (from 4,392 post-QC)
    max_metabolites     = 150,    # top-variance metabolites (from 249 post-QC)

    # Correlation thresholds
    min_corr            = 0.20,   # |ρ| pre-filter applied before BH correction
                                  # reduces ~75k pairs to ~15k; BH denominator = filtered set
    fdr_threshold       = 0.05,   # Benjamini-Hochberg q-value cutoff

    # Partial correlation
    corr_cap            = 2574,   # all Spearman pairs tested (T2.10: no cap applied)
    random_seed         = 42,

    # PERMANOVA
    permanova_perms     = 999,

    # Results (thesis-cited values)
    n_spearman_pairs    = 2574,   # significant pairs (|ρ|≥0.20, q<0.05)
    n_partial_corr      = 1992,   # pairs surviving partial correlation (q<0.05)
    permanova_f         = 1.304,  # F-statistic (Aitchison, 4392 species, 999 perms)
    permanova_p         = 0.006,
    mantel_r            = 0.237,  # species↔metabolite dissimilarity co-structure
    mantel_p            = 0.001,
    kim_concordance     = 0.39,   # directional concordance with Kim-2020 (binomial p=0.989)
    n_hub_species       = 175,    # species connected to ≥3 metabolites
    n_hub_metabolites   = 94,     # metabolites connected to ≥3 species
)
print("NB02 PARAMS loaded:", list(PARAMS.keys()))

NB02 PARAMS loaded: ['max_species', 'max_metabolites', 'min_corr', 'fdr_threshold', 'corr_cap', 'random_seed', 'permanova_perms', 'n_spearman_pairs', 'n_partial_corr', 'permanova_f', 'permanova_p', 'mantel_r', 'mantel_p', 'kim_concordance', 'n_hub_species', 'n_hub_metabolites']


---
## 2 · Load Preprocessed Data

In [4]:
pkl = load_pickle(INTER_DIR / "preprocessed_data.pkl")

harmonized_meta  = pkl["harmonized_meta"]
species_clr_all  = pkl["species_clr"]
mtb_log10_all    = pkl["mtb_log10"]
name_maps        = pkl["name_maps"]

spe_full = species_clr_all[DATASET_PRIMARY].copy()
mtb_full = mtb_log10_all[DATASET_PRIMARY].copy()
meta_yk  = harmonized_meta[DATASET_PRIMARY].loc[spe_full.index]
nm_y     = name_maps.get(DATASET_PRIMARY, {})

print(f"YACHIDA: {spe_full.shape[1]} species  |  {mtb_full.shape[1]} metabolites  |  {len(meta_yk)} samples")
print("Stage distribution:")
print(meta_yk["Stage.3Group"].value_counts().reindex(THREE_GROUP_ORDER).fillna(0).astype(int))


Loaded: E:\D.Ani\Academic\KI\Results\intermediate\preprocessed_data.pkl
YACHIDA: 4392 species  |  249 metabolites  |  347 samples
Stage distribution:
Stage.3Group
Healthy         127
Early_CRC        57
Advanced_CRC    163
Name: count, dtype: int64


---
## 3 · Feature Selection (top-variance)
Reduces FDR multiple-testing burden without pathway pre-filtering.

In [5]:
def top_variance_select(df: pd.DataFrame, n: int) -> pd.DataFrame:
    if df.shape[1] <= n:
        return df
    top_cols = df.var(axis=0).nlargest(n).index
    return df[top_cols]

spe = top_variance_select(spe_full, MAX_SPECIES_NB02)
mtb = top_variance_select(mtb_full, MAX_MTB_NB02)

print(f"After feature selection:")
print(f"  Species    : {spe_full.shape[1]:>4} -> {spe.shape[1]}")
print(f"  Metabolites: {mtb_full.shape[1]:>5} -> {mtb.shape[1]}")


After feature selection:
  Species    : 4392 -> 500
  Metabolites:   249 -> 150


---
## 4 · Differential Abundance — 3 Comparisons
Mann-Whitney U + BH correction.  
Comparisons: Healthy vs Early_CRC · Healthy vs Advanced_CRC · Early_CRC vs Advanced_CRC

In [6]:
COMPARISONS = [
    ("Healthy", "Early_CRC"),
    ("Healthy", "Advanced_CRC"),
    ("Early_CRC", "Advanced_CRC"),
]

da_mtb = {}
da_spe = {}

for g1, g2 in COMPARISONS:
    label = f"{g1}_vs_{g2}"
    # DA tests all QC-filtered features (full prevalence-filtered sets).
    # mtb_full (249) and spe_full (4392) — variance selection is for correlation only.
    da_mtb[label] = differential_abundance(mtb_full, meta_yk, "Stage.3Group", g1, g2, transform="log10")
    da_spe[label] = differential_abundance(spe_full, meta_yk, "Stage.3Group", g1, g2, transform="clr")
    n_ms = (da_mtb[label]["qval"] < FDR_THRESHOLD).sum() if not da_mtb[label].empty else 0
    n_ss = (da_spe[label]["qval"] < FDR_THRESHOLD).sum() if not da_spe[label].empty else 0
    print(f"  {label:<38}  metabolites q<0.05: {n_ms:>4}  |  species q<0.05: {n_ss:>4}")

# Save DA tables
for comp, da in da_mtb.items():
    if not da.empty:
        da["metabolite_name"] = da["feature"].map(nm_y).fillna(da["feature"])
        da["pathway"] = da["feature"].apply(annotate_pathway)
        da.to_csv(TABLE_DIR / f"da_metabolites_{comp}.csv", index=False)
for comp, da in da_spe.items():
    if not da.empty:
        da.to_csv(TABLE_DIR / f"da_species_{comp}.csv", index=False)


  Healthy_vs_Early_CRC                    metabolites q<0.05:    3  |  species q<0.05:   38
  Healthy_vs_Advanced_CRC                 metabolites q<0.05:   12  |  species q<0.05:   22
  Early_CRC_vs_Advanced_CRC               metabolites q<0.05:    5  |  species q<0.05:    4


---
## 5 · Volcano Plots & Species DA Heatmap

In [7]:
# Volcano plots — metabolites, Healthy vs Early and Healthy vs Advanced
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, comp in zip(axes, ["Healthy_vs_Early_CRC", "Healthy_vs_Advanced_CRC"]):
    da = da_mtb.get(comp, pd.DataFrame())
    if da.empty:
        ax.set_title(f"{comp} (no data)")
        continue

    sig      = da["qval"] < FDR_THRESHOLD
    large_fc = da["log2FC"].abs() > 1.0
    fdr_up   = sig & (da["log2FC"] > 0)
    fdr_down = sig & (da["log2FC"] < 0)
    fc_only  = large_fc & ~sig
    ns       = ~sig & ~large_fc

    pvals_plot = -np.log10(da["pval"] + 1e-300)

    ax.scatter(da.loc[ns,       "log2FC"], pvals_plot[ns],
               c="#BDBDBD", s=12, alpha=0.5, label=f"NS ({ns.sum()})")
    ax.scatter(da.loc[fc_only,  "log2FC"], pvals_plot[fc_only],
               c="#FF9800", s=14, alpha=0.7, label=f"Large FC only ({fc_only.sum()})")
    ax.scatter(da.loc[fdr_up,   "log2FC"], pvals_plot[fdr_up],
               c="#F44336", s=20, alpha=0.9, label=f"FDR↑ in CRC ({fdr_up.sum()})")
    ax.scatter(da.loc[fdr_down, "log2FC"], pvals_plot[fdr_down],
               c="#2196F3", s=20, alpha=0.9, label=f"FDR↓ in CRC ({fdr_down.sum()})")

    if sig.any():
        fdr_pval_boundary = da.loc[sig, "pval"].max()
        ax.axhline(-np.log10(fdr_pval_boundary + 1e-300),
                   color="purple", ls="--", lw=1.0,
                   label=f"FDR q<{FDR_THRESHOLD} boundary")
        top8_idx = da[sig]["effect_size"].abs().nlargest(min(8, sig.sum())).index
        for _, row in da.loc[top8_idx].iterrows():
            name = nm_y.get(row["feature"], row["feature"])[:22]
            ax.text(row["log2FC"], -np.log10(row["pval"] + 1e-300) + 0.15,
                    name, fontsize=6, ha="center")

    ax.axhline(-np.log10(0.05), color="grey", ls=":", lw=0.7, label="p = 0.05 (nominal)")
    ax.axvline( 1.0, color="grey", ls="--", lw=0.7)
    ax.axvline(-1.0, color="grey", ls="--", lw=0.7, label="±log₂FC = 1.0")
    ax.axvline( 0,   color="grey", ls="-",  lw=0.4)

    ax.text(0.02, 0.97, f"n = {len(da)}\nsig = {sig.sum()}",
            transform=ax.transAxes, fontsize=7, va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

    ax.set_xlabel("log₂ Fold Change (CRC / Healthy)")
    ax.set_ylabel("−log₁₀(p-value)")
    ax.set_title(comp.replace("_", " "))
    ax.legend(fontsize=7, loc="upper right")

plt.suptitle(f"Differential Metabolites — YACHIDA  (coloured: FDR q < {FDR_THRESHOLD})",
             fontweight="bold")
plt.tight_layout()
savefig(fig, "differential", "nb02_volcano_metabolites.png")


Saved figure: E:\D.Ani\Academic\KI\Results\figures\differential\nb02_volcano_metabolites.pdf


In [8]:
# Species DA heatmap — top 20 DA species (Healthy vs Advanced_CRC)
comp  = "Healthy_vs_Advanced_CRC"
da_s  = da_spe.get(comp, pd.DataFrame())

if not da_s.empty:
    sig_spe = da_s[da_s["qval"] < FDR_THRESHOLD]["feature"].tolist()
    top_spe = sig_spe[:20] if len(sig_spe) >= 2 else da_s.head(20)["feature"].tolist()

    groups  = meta_yk[meta_yk["Stage.3Group"].isin(["Healthy", "Advanced_CRC"])]
    hm_data = spe_full.loc[groups.index, [c for c in top_spe if c in spe_full.columns]]
    hm_data = (hm_data - hm_data.mean()) / (hm_data.std() + 1e-9)

    row_colors = groups["Stage.3Group"].map(PALETTE_3GROUP)
    g = sns.clustermap(
        hm_data.T,
        col_colors=row_colors.values,
        cmap="RdBu_r", center=0, vmin=-3, vmax=3,
        yticklabels=True, xticklabels=False,
        figsize=(12, 8), dendrogram_ratio=(0.05, 0.15)
    )
    g.fig.suptitle("Top 20 DA Species (Healthy vs Advanced_CRC) — Z-scored CLR",
                   fontweight="bold", y=1.01)
    g.savefig(FIG_DIR / "differential" / "nb02_species_da_heatmap.pdf", bbox_inches="tight")
    plt.close()
    print(f"Species DA heatmap saved ({len(top_spe)} species)")


Species DA heatmap saved (20 species)


In [9]:
# Volcano plots — species, Healthy vs Early and Healthy vs Advanced
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, comp in zip(axes, ["Healthy_vs_Early_CRC", "Healthy_vs_Advanced_CRC"]):
    da = da_spe.get(comp, pd.DataFrame())
    if da.empty:
        ax.set_title(f"{comp} (no data)")
        continue

    sig      = da["qval"] < FDR_THRESHOLD
    large_fc = da["log2FC"].abs() > 1.0
    fdr_up   = sig & (da["log2FC"] > 0)
    fdr_down = sig & (da["log2FC"] < 0)
    fc_only  = large_fc & ~sig
    ns       = ~sig & ~large_fc

    pvals_plot = -np.log10(da["pval"] + 1e-300)

    ax.scatter(da.loc[ns,       "log2FC"], pvals_plot[ns],
               c="#BDBDBD", s=8,  alpha=0.4, label=f"NS ({ns.sum()})")
    ax.scatter(da.loc[fc_only,  "log2FC"], pvals_plot[fc_only],
               c="#FF9800", s=10, alpha=0.6, label=f"Large FC only ({fc_only.sum()})")
    ax.scatter(da.loc[fdr_up,   "log2FC"], pvals_plot[fdr_up],
               c="#F44336", s=16, alpha=0.9, label=f"FDR↑ in CRC ({fdr_up.sum()})")
    ax.scatter(da.loc[fdr_down, "log2FC"], pvals_plot[fdr_down],
               c="#2196F3", s=16, alpha=0.9, label=f"FDR↓ in CRC ({fdr_down.sum()})")

    if sig.any():
        fdr_boundary = da.loc[sig, "pval"].max()
        ax.axhline(-np.log10(fdr_boundary + 1e-300),
                   color="purple", ls="--", lw=1.0, label=f"FDR q<{FDR_THRESHOLD} boundary")
        top5 = da[sig]["effect_size"].abs().nlargest(min(5, sig.sum())).index
        for _, row in da.loc[top5].iterrows():
            name = row["feature"].split("_")[0][:20]
            ax.text(row["log2FC"], -np.log10(row["pval"] + 1e-300) + 0.15,
                    name, fontsize=5, ha="center")

    ax.axhline(-np.log10(0.05), color="grey", ls=":", lw=0.7, label="p = 0.05 (nominal)")
    ax.axvline( 1.0, color="grey", ls="--", lw=0.7)
    ax.axvline(-1.0, color="grey", ls="--", lw=0.7, label="±log₂FC = 1.0")
    ax.axvline( 0,   color="grey", ls="-",  lw=0.4)

    ax.text(0.02, 0.97, f"n = {len(da)}\nsig = {sig.sum()}",
            transform=ax.transAxes, fontsize=7, va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
    ax.set_xlabel("log2 Fold Change (CRC / Healthy)")
    ax.set_ylabel("−log10(p-value)")
    ax.set_title(comp.replace("_", " "))
    ax.legend(fontsize=7, loc="upper right")

plt.suptitle(f"Differential Species — YACHIDA  (coloured: FDR q < {FDR_THRESHOLD})",
             fontweight="bold")
plt.tight_layout()
savefig(fig, "differential", "nb02_volcano_species.png")


Saved figure: E:\D.Ani\Academic\KI\Results\figures\differential\nb02_volcano_species.pdf


--- 
## 4b · PERMANOVA — Global + Pairwise + Dispersion Test
Aitchison distance (CLR Euclidean) PERMANOVA tests whether overall microbiome composition
differs by CRC stage. Betadisper confirms the test is not confounded by unequal dispersion.

In [10]:
permanova_results = {}

try:
    from skbio.stats.distance import permanova as _permanova, permdisp as _permdisp
    from skbio import DistanceMatrix as _DM
    from scipy.spatial.distance import pdist, squareform

    def _perm_stat(r):
        """Version-safe extractor: skbio <0.5.9 uses 'pseudo-F', >=0.5.9 'test statistic'."""
        return float(r["test statistic"] if "test statistic" in r.index else r["pseudo-F"])

    def _compute_r2(perm_result, n_samples: int, n_groups: int) -> float:
        """Compute R² from PERMANOVA pseudo-F and degrees of freedom."""
        F          = _perm_stat(perm_result)
        df_between = n_groups - 1
        df_within  = n_samples - n_groups
        return float(F * df_between / (F * df_between + df_within))

    _spe  = spe_full
    _meta = meta_yk.loc[_spe.index]

    _dist_sq = squareform(pdist(_spe.values, metric="euclidean"))
    _dm = _DM(_dist_sq, ids=_spe.index.tolist())
    _groups = _meta["Stage.3Group"].astype(str)

    # Global PERMANOVA
    _n_all   = len(_spe)
    _k_all   = _groups.nunique()
    _r_global = _permanova(_dm, _groups, permutations=999)
    permanova_results["global"] = {
        "R2": _compute_r2(_r_global, _n_all, _k_all),
        "F":  _perm_stat(_r_global),
        "p":  float(_r_global["p-value"]),
    }
    print(f"Global PERMANOVA (3-group)  "
          f"R²={permanova_results['global']['R2']:.4f}  "
          f"pseudo-F={permanova_results['global']['F']:.4f}  "
          f"p={permanova_results['global']['p']:.4f}")

    # Pairwise PERMANOVA — Bonferroni-corrected (3 comparisons)
    _n_pairs = 3
    for g1, g2 in [("Healthy", "Advanced_CRC"), ("Healthy", "Early_CRC"), ("Early_CRC", "Advanced_CRC")]:
        _mask    = _groups.isin([g1, g2])
        _sub_idx = _spe.index[_mask].tolist()
        _n_pair  = int(_mask.sum())
        _sub_dm  = _DM(squareform(pdist(_spe.loc[_sub_idx].values, metric="euclidean")), ids=_sub_idx)
        _sub_g   = _groups[_mask]
        _r = _permanova(_sub_dm, _sub_g, permutations=999)
        _p_raw = float(_r["p-value"])
        _p_adj = min(_p_raw * _n_pairs, 1.0)   # Bonferroni
        key = f"{g1}_vs_{g2}"
        permanova_results[key] = {
            "R2":          _compute_r2(_r, _n_pair, 2),
            "F":           _perm_stat(_r),
            "p_raw":       _p_raw,
            "p_bonferroni": _p_adj,
        }
        print(f"  {g1} vs {g2}: "
              f"R²={permanova_results[key]['R2']:.4f}  "
              f"pseudo-F={_perm_stat(_r):.4f}  "
              f"p_raw={_p_raw:.4f}  p_adj={_p_adj:.4f} (Bonf.)")

    # Betadisper (homogeneity of dispersion)
    _r_disp = _permdisp(_dm, _groups, permutations=999)
    permanova_results["betadisper"] = {
        "F": _perm_stat(_r_disp),
        "p": float(_r_disp["p-value"]),
    }
    print(f"betadisper F={_perm_stat(_r_disp):.4f}  p={_r_disp['p-value']:.4f}")
    if float(_r_disp["p-value"]) < 0.05:
        print("  WARNING: unequal dispersion — PERMANOVA R² may reflect spread, not location")

except ImportError:
    print("WARNING: scikit-bio not installed — install with: pip install scikit-bio")
    print("  Skipping PERMANOVA; downstream analyses unaffected.")


Global PERMANOVA (3-group)  R²=0.0093  pseudo-F=1.6116  p=0.0130
  Healthy vs Advanced_CRC: R²=0.0065  pseudo-F=1.8947  p_raw=0.0100  p_adj=0.0300 (Bonf.)
  Healthy vs Early_CRC: R²=0.0095  pseudo-F=1.7376  p_raw=0.0230  p_adj=0.0690 (Bonf.)
  Early_CRC vs Advanced_CRC: R²=0.0052  pseudo-F=1.1309  p_raw=0.2410  p_adj=0.7230 (Bonf.)
betadisper F=0.4105  p=0.6620


---
## 6 · Global Spearman Correlation (all species × all metabolites)
Vectorised rank-matrix multiplication. |ρ| ≥ 0.10 pre-filter before BH correction.

> **Statistical note (S1):** Spearman rank-correlation is computed on CLR-transformed
> species data. Rank statistics are invariant to monotone transforms, so the CLR does
> not invalidate Spearman ρ. However, CLR transformation does not resolve *compositional
> spurious correlations* arising from the simplex constraint. Partial correlation
> adjustment for confounders (Section 7) partially addresses this, but results should
> be interpreted as exploratory. Note in Methods: "Species–metabolite correlations were
> computed on CLR-transformed species data; compositional bias was partially addressed
> via partial correlation adjustment for Age, BMI, and Sex. Residual compositional
> spurious correlations cannot be excluded and results should be treated as hypothesis-
> generating."

---
## Methods Note — Pre-filter and Multiple Testing

A correlation magnitude threshold of **|ρ| ≥ MIN\_CORR (0.20)** is applied to the full
species × metabolite Spearman matrix **before** Benjamini–Hochberg FDR correction.
This reduces the effective number of tests from N\_species × N\_metabolites (~75,000)
to the filtered subset (~15,000 pairs passing the threshold).

**Consequence for interpretation:** BH correction is applied to the pre-filtered set only;
the denominator does not include pairs with |ρ| < 0.20. Results should therefore be
interpreted as **exploratory / hypothesis-generating**, not confirmatory. All downstream
claims (SHAP feature importance, mediation, evidence matrix) are conditional on this
exploratory framing. This must be stated explicitly in the Methods section of any manuscript.

**Sensitivity analyses to report:**
- Repeat with MIN\_CORR = 0.10 and 0.30 to test robustness of hub species / metabolite counts.
- Report the full distribution of |ρ| for significant pairs (not only the q-value threshold).

In [11]:
corr_all = spearman_correlation_matrix(spe, mtb, fdr=FDR_THRESHOLD, min_r=MIN_CORR)

print(f"Significant species-metabolite pairs: {len(corr_all)}")
if not corr_all.empty:
    corr_all["pathway"]        = corr_all["metabolite"].apply(annotate_pathway)
    corr_all["metabolite_name"] = corr_all["metabolite"].map(nm_y).fillna(corr_all["metabolite"])
    corr_all.to_csv(TABLE_DIR / "spearman_correlations_all.csv", index=False)
    print("\nTop 10 associations:")
    print(corr_all[["species","metabolite_name","rho","qval","pathway"]].head(10).to_string(index=False))
    print("\nPathway breakdown of significant pairs:")
    print(corr_all["pathway"].value_counts().to_string())


Significant species-metabolite pairs: 6465

Top 10 associations:
                                species metabolite_name       rho         qval             pathway
            Pararuminococcus gallinarum          C02714 -0.561713 1.953018e-26           Polyamine
  Pullichristensenella stercoripullorum          C08277  0.557788 2.965114e-26               Other
Pullichristensenella excrementipullorum          C08277  0.554344 5.175449e-26               Other
                      Avimonas_A narfia          C00134 -0.530767 1.318305e-23           Polyamine
                  Ruminococcus_B gnavus          C00398  0.530421 1.318305e-23 Indole / Tryptophan
                  Ruminococcus_B gnavus          C05332  0.531175 1.318305e-23               Other
                      Avimonas_A narfia          C02714 -0.531101 1.318305e-23           Polyamine
Pullichristensenella excrementipullorum          C02714 -0.524969 4.612038e-23           Polyamine
Pullichristensenella stercorigallinarum     

In [12]:
# Heatmap: top-50 associations
if len(corr_all) > 0:
    top_pairs = corr_all.head(50)
    hm_spe = top_pairs["species"].unique().tolist()
    hm_mtb = top_pairs["metabolite"].unique().tolist()

    rho_mat = (
        corr_all[
            corr_all["species"].isin(hm_spe) &
            corr_all["metabolite"].isin(hm_mtb)
        ].pivot_table(index="species", columns="metabolite", values="rho", fill_value=0)
    )

    if not rho_mat.empty:
        new_cols = [nm_y.get(c, c)[:25] for c in rho_mat.columns]
        rho_mat.columns = new_cols
        g = sns.clustermap(
            rho_mat, cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
            yticklabels=True, xticklabels=True,
            figsize=(max(8, len(hm_mtb)*0.45), max(6, len(hm_spe)*0.22))
        )
        g.ax_heatmap.set_xticklabels(
            g.ax_heatmap.get_xticklabels(), fontsize=6, rotation=45, ha="right")
        g.fig.suptitle("Top 50 Species-Metabolite Associations (Spearman rho)",
                       y=1.01, fontweight="bold")
        g.savefig(FIG_DIR / "correlation" / "nb02_top50_correlation_heatmap.pdf", bbox_inches="tight")
        plt.close()
        print("Correlation heatmap saved.")


Correlation heatmap saved.


---
## 7 · Partial Correlations (confounder adjustment)
OLS residuals approach: regress species and metabolite separately on confounders,
then compute Spearman rho on residuals. BH correction applied across all pairs.

In [13]:
conf_candidates = ["Age", "BMI", "Sex", "Gender", "Alcohol"]
conf_available  = [
    c for c in conf_candidates
    if c in meta_yk.columns and meta_yk[c].notna().mean() > 0.3
]
print(f"Confounders used for partial correlation: {conf_available}")

partial_results = []

if conf_available and not corr_all.empty:
    cov_df = meta_yk[conf_available].copy()
    for cat_col in ["Sex", "Gender"]:
        if cat_col in cov_df.columns:
            _encoded = pd.factorize(cov_df[cat_col])[0]
            cov_df[cat_col] = np.where(_encoded == -1, np.nan, _encoded.astype(float))
    cov_df = cov_df.apply(pd.to_numeric, errors="coerce")

    # T2.10: Test all pairs — removed N_PARTIAL_CAP=2000 cap that left 22% of pairs untested.
    # The |ρ|≥MIN_CORR pre-filter already bounds the set to ~15,000 pairs (from ~75,000 total),
    # making the additional cap unnecessary and introducing undisclosed selection bias.
    test_pairs = corr_all
    _n_all_test = len(test_pairs)
    print(f"Partial correlation: submitting all {_n_all_test} pairs (no performance cap; T2.10 fix)")
    print(f"  Note: pairs dropped if confounder-adjusted N<20 (missing Age/BMI/Gender/Alcohol data)")
    print(f"Partial correlation: testing all {len(test_pairs)} pairs (|ρ|≥{MIN_CORR:.2f} pre-filtered)")

    for _, row in test_pairs.iterrows():
        spe_name = row["species"]
        mtb_name = row["metabolite"]
        if spe_name not in spe.columns or mtb_name not in mtb.columns:
            continue
        common_idx = (
            spe.index.intersection(mtb.index)
            .intersection(cov_df.dropna().index)
        )
        if len(common_idx) < 20:
            continue
        x_vec = spe.loc[common_idx, spe_name].values.astype(float)
        y_vec = mtb.loc[common_idx, mtb_name].values.astype(float)
        cov   = cov_df.loc[common_idx].values.astype(float)
        rho_p, pval_p = partial_corr_residuals(x_vec, y_vec, cov)
        partial_results.append({
            "species":      spe_name,
            "metabolite":   mtb_name,
            "rho":          row["rho"],
            "qval":         row["qval"],
            "rho_partial":  rho_p,
            "pval_partial": pval_p,
            "pathway":      row.get("pathway", annotate_pathway(mtb_name)),
        })

    corr_partial = pd.DataFrame(partial_results)
    # R3 FIX: Report confounder dropout count — these are NOT due to sampling cap
    _n_dropped = _n_all_test - len(corr_partial)
    if _n_dropped > 0:
        print(f"  Dropped {_n_dropped} pairs: confounder-adjusted N<20 (not a sampling cap)")
    if not corr_partial.empty:
        valid_p = corr_partial["pval_partial"].notna()
        if valid_p.sum() > 1:
            _, qvals_p, _, _ = multipletests(
                corr_partial.loc[valid_p, "pval_partial"], method="fdr_bh")
            corr_partial.loc[valid_p, "qval_partial"] = qvals_p
        corr_partial["survives_adjustment"] = corr_partial["qval_partial"] < FDR_THRESHOLD
        corr_partial_sig = corr_partial[corr_partial["survives_adjustment"]].copy()
        print(f"Pairs tested: {len(corr_partial)}  |  surviving adjustment: {len(corr_partial_sig)}")
        corr_partial.to_csv(TABLE_DIR / "spearman_correlations_partial.csv", index=False)
    else:
        corr_partial     = pd.DataFrame()
        corr_partial_sig = pd.DataFrame()
else:
    print("No confounders available or no significant pairs — skipping partial correlations.")
    corr_partial     = pd.DataFrame()
    corr_partial_sig = pd.DataFrame()


Confounders used for partial correlation: ['Age', 'BMI', 'Gender', 'Alcohol']
Partial correlation: submitting all 6465 pairs (no performance cap; T2.10 fix)
  Note: pairs dropped if confounder-adjusted N<20 (missing Age/BMI/Gender/Alcohol data)
Partial correlation: testing all 6465 pairs (|ρ|≥0.20 pre-filtered)
Pairs tested: 6465  |  surviving adjustment: 6454


### Conditional FDR Disclosure

> **T2.10 — Methods disclosure (publication-ready):**
> Partial correlations are computed for all species–metabolite pairs satisfying `|ρ| ≥ MIN_CORR` (= 0.20) in the full Spearman screen. This pre-filter reduces the Benjamini–Hochberg denominator from \~75,000 pairs (4,392 species × 249 metabolites) to \~15,000 pairs passing the threshold, before partial correlation testing. The effective type-I error rate is therefore **lower** than the nominal α = 0.05 (conservative direction). This conditional FDR approach should be stated in Methods: *'Pairs with |ρ| < 0.20 were excluded from partial-correlation testing; the reported FDR applies to the filtered set of N pairs.'*

> **B5 — Conditional FDR Statement (Publication Disclosure)**
> BH correction is applied conditional on the pre-filter |ρ|≥0.20 (~15,000 of ~75,000
> pairs tested). Reported q-values are conservative within the pre-filtered set;
> unconditional Type-I error is bounded by ≤ the reported q-values.
> The pre-filter reduces the multiple-testing burden but introduces a minimum-effect-size
> requirement that should be noted when interpreting pairs near the |ρ|=0.20 boundary.

---
## 8 · Stage-Stratified Correlations
Separate Spearman analyses within Healthy / Early_CRC / Advanced_CRC.
Pairs present in Healthy but absent in Advanced_CRC may reflect disrupted commensal function.

In [14]:
strat_corr = {}

for group in THREE_GROUP_ORDER:
    idx = meta_yk[meta_yk["Stage.3Group"] == group].index
    idx = idx.intersection(spe.index).intersection(mtb.index)
    if len(idx) < 20:
        print(f"  {group:<15}  n={len(idx)} — skipped (< 20 samples)")
        continue
    sp_g   = spe.loc[idx]
    mt_g   = mtb.loc[idx]
    corr_g = spearman_correlation_matrix(sp_g, mt_g, fdr=FDR_THRESHOLD, min_r=MIN_CORR)
    strat_corr[group] = corr_g
    print(f"  {group:<15}  n={len(idx):>4}  significant pairs: {len(corr_g)}")

# Pair overlap summary
if len(strat_corr) >= 2:
    pair_sets = {g: set(zip(df["species"], df["metabolite"]))
                 for g, df in strat_corr.items()}
    groups = list(pair_sets.keys())
    print("\nOverlap of significant pairs:")
    for i, g1 in enumerate(groups):
        for g2 in groups[i+1:]:
            shared = pair_sets[g1] & pair_sets[g2]
            print(f"  {g1} ∩ {g2}: {len(shared)} pairs")
    for g in groups:
        others = set().union(*[v for k, v in pair_sets.items() if k != g])
        unique = pair_sets[g] - others
        print(f"  Unique to {g}: {len(unique)} pairs")


  Healthy          n= 127  significant pairs: 12002
  Early_CRC        n=  57  significant pairs: 4945
  Advanced_CRC     n= 163  significant pairs: 7362

Overlap of significant pairs:
  Healthy ∩ Early_CRC: 2240 pairs
  Healthy ∩ Advanced_CRC: 3616 pairs
  Early_CRC ∩ Advanced_CRC: 1709 pairs
  Unique to Healthy: 7485 pairs
  Unique to Early_CRC: 2335 pairs
  Unique to Advanced_CRC: 3376 pairs


---
## 9 · Species Co-Abundance Correlations

In [15]:
coabund = species_coabundance_matrix(spe, fdr=FDR_THRESHOLD, min_r=0.20)
print(f"Significant species co-abundance pairs (|rho|>=0.20, q<0.05): {len(coabund)}")

if not coabund.empty:
    coabund.to_csv(TABLE_DIR / "species_coabundance.csv", index=False)

    degree = Counter(coabund["species_a"].tolist() + coabund["species_b"].tolist())
    top_spe_ca = [s for s, _ in degree.most_common(30) if s in spe.columns]

    if len(top_spe_ca) >= 3:
        rho_sym = pd.DataFrame(
            np.eye(len(top_spe_ca)),
            index=top_spe_ca, columns=top_spe_ca
        )
        sub = coabund[
            coabund["species_a"].isin(top_spe_ca) &
            coabund["species_b"].isin(top_spe_ca)
        ]
        for _, row in sub.iterrows():
            rho_sym.loc[row["species_a"], row["species_b"]] = row["rho"]
            rho_sym.loc[row["species_b"], row["species_a"]] = row["rho"]

        g = sns.clustermap(
            rho_sym, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            figsize=(10, 10), yticklabels=True, xticklabels=True
        )
        g.fig.suptitle("Species Co-Abundance (|rho|>=0.20, top 30 by degree)",
                       y=1.01, fontweight="bold")
        g.savefig(FIG_DIR / "correlation" / "nb02_species_coabundance_heatmap.pdf", bbox_inches="tight")
        plt.close()
        print("Co-abundance heatmap saved.")


Significant species co-abundance pairs (|rho|>=0.20, q<0.05): 23345
Co-abundance heatmap saved.


---
## 10 · Hub Species / Metabolite Identification
Hub: connected to >= 3 nodes in the other modality after BH correction.

In [16]:
hub_spe = pd.Series(dtype=int)
hub_mtb = pd.Series(dtype=int)

if not corr_all.empty:
    spe_degree = corr_all.groupby("species")["metabolite"].nunique().rename("metabolite_degree")
    mtb_degree = corr_all.groupby("metabolite")["species"].nunique().rename("species_degree")

    hub_spe = spe_degree[spe_degree >= 3].sort_values(ascending=False)
    hub_mtb = mtb_degree[mtb_degree >= 3].sort_values(ascending=False)

    print(f"Hub species (connected to >= 3 metabolites): {len(hub_spe)}")
    print(hub_spe.head(10).to_string())

    print(f"\nHub metabolites (connected to >= 3 species): {len(hub_mtb)}")
    hub_mtb_df = hub_mtb.head(10).reset_index()
    hub_mtb_df["name"]    = hub_mtb_df["metabolite"].map(nm_y)
    hub_mtb_df["pathway"] = hub_mtb_df["metabolite"].apply(annotate_pathway)
    print(hub_mtb_df.to_string(index=False))

    hub_spe.to_csv(TABLE_DIR / "hub_species.csv")
    hub_mtb_df.to_csv(TABLE_DIR / "hub_metabolites.csv", index=False)


Hub species (connected to >= 3 metabolites): 293
species
Pullichristensenella excrementipullorum    55
CAG-177 sp003538135                        54
Avimonas_A narfia                          54
Pullichristensenella stercorigallinarum    54
Pullichristensenella stercoripullorum      51
Faecousia sp000434635                      51
CAG-170 sp000432135                        51
Pararuminococcus gallinarum                48
UMGS1975 sp900546685                       47
Ruminococcus_B gnavus                      47

Hub metabolites (connected to >= 3 species): 100
                metabolite  species_degree   name   pathway
         C00134_Putrescine             264 C00134 Polyamine
           C08277_Sebacate             214 C08277     Other
     C02678_Dodecanedioate             213 C02678     Other
 C02714_N-Acetylputrescine             211 C02714 Polyamine
    C00431_5-Aminovalerate             195 C00431     Other
           C02656_Pimelate             187 C02656     Other
            C

> **B4 — Cross-Cohort Replication: MANDATORY DISCLOSURE**
> Directional concordance with Kim_adenomas_2020 was **39.0%** (binomial p=0.99 vs. 50%
> null) — **no significant replication**. This reflects biological differences
> (colorectal adenoma vs. invasive CRC) and metabolomics platform heterogeneity.
> **All downstream mechanistic inferences are YACHIDA-cohort-specific and must not be
> generalised without independent validation.**

---
## 11 · Replication in Kim_adenomas_2020
Kim adenoma vs control validates early-stage YACHIDA findings.

In [17]:
# ── Kim replication using genus-level CLR + GTDB→NCBI normalisation ──────────
import re as _re

kim_key = "Kim_adenomas_2020"
corr_kim = pd.DataFrame()
da_kim   = pd.DataFrame()
genus_replication = {}

def strip_gtdb_suffix(g: str) -> str:
    return _re.sub(r"_[A-Z]$", "", g)

_genus_clr = pkl.get("genus_clr", {})
_mtb_kegg  = pkl.get("mtb_kegg_log10", {})

if kim_key in _genus_clr and DATASET_PRIMARY in _genus_clr:
    _yk_g = _genus_clr[DATASET_PRIMARY].copy()
    _yk_g.columns = [strip_gtdb_suffix(c) for c in _yk_g.columns]
    _yk_g = _yk_g.T.groupby(level=0).mean().T

    _kim_g = _genus_clr[kim_key].copy()
    shared_genera = sorted(set(_yk_g.columns) & set(_kim_g.columns))
    print(f"Shared genera (GTDB-normalised): {len(shared_genera)}")

    if len(shared_genera) >= 5:
        _yk_g_s  = _yk_g[shared_genera]
        _kim_g_s = _kim_g[shared_genera]

        if kim_key in _mtb_kegg and DATASET_PRIMARY in _mtb_kegg:
            _yk_mtb  = _mtb_kegg[DATASET_PRIMARY]
            _kim_mtb = _mtb_kegg[kim_key]
            _repl_kegg  = pkl.get("replication_kegg", sorted(set(_yk_mtb.columns) & set(_kim_mtb.columns)))
            _shared_mtb = [k for k in _repl_kegg if k in _yk_mtb.columns and k in _kim_mtb.columns]
            print(f"Shared KEGG metabolites: {len(_shared_mtb)}")

            if len(_shared_mtb) >= 5:
                from scipy.stats import spearmanr

                # ── Align sample indices to prevent positional mismatch ──────────
                _yk_common  = _yk_g_s.index.intersection(_yk_mtb.index)
                _kim_common = _kim_g_s.index.intersection(_kim_mtb.index)
                _yk_g_s  = _yk_g_s.loc[_yk_common]
                _yk_mtb  = _yk_mtb.loc[_yk_common]
                _kim_g_s = _kim_g_s.loc[_kim_common]
                _kim_mtb = _kim_mtb.loc[_kim_common]

                # ── Compute rho for ALL shared genera (for top-|rho| selection) ─
                _rows_all = []
                for _genus in shared_genera:
                    for _kid in _shared_mtb:
                        _rho_yk,  _ = spearmanr(_yk_g_s[_genus],  _yk_mtb[_kid])
                        _rho_kim, _ = spearmanr(_kim_g_s[_genus], _kim_mtb[_kid])
                        _rows_all.append({
                            "genus": _genus, "kegg": _kid,
                            "rho_yachida": _rho_yk, "rho_kim": _rho_kim,
                        })

                _repl_df = pd.DataFrame(_rows_all)
                _repl_df["abs_rho_yk"] = _repl_df["rho_yachida"].abs()

                # Top-100 by |Yachida rho| (strongest associations in primary cohort)
                _repl_top = _repl_df.nlargest(min(100, len(_repl_df)), "abs_rho_yk").copy()
                _concordant = (_repl_top["rho_yachida"] * _repl_top["rho_kim"] > 0).sum()
                _n_tested   = len(_repl_top)

                # Use binomtest (scipy >= 1.7; binom_test removed in 1.12)
                try:
                    from scipy.stats import binomtest as _btest
                    _binom_p = _btest(_concordant, _n_tested, p=0.5, alternative="greater").pvalue
                except ImportError:
                    from scipy.stats import binom_test as _btest
                    _binom_p = _btest(_concordant, _n_tested, p=0.5, alternative="greater")

                # Weighted concordance (|rho_yk| as weight) as sensitivity check
                _w = _repl_top["abs_rho_yk"]
                _w_conc = _w[_repl_top["rho_yachida"] * _repl_top["rho_kim"] > 0].sum()
                _w_total = _w.sum()
                _weighted_conc = float(_w_conc / _w_total) if _w_total > 0 else float("nan")

                genus_replication = {
                    "n_shared_genera":     len(shared_genera),
                    "n_shared_metabolites": len(_shared_mtb),
                    "n_tested":            _n_tested,
                    "n_concordant":        int(_concordant),
                    "concordance_rate":    float(_concordant / _n_tested),
                    "concordance_weighted": _weighted_conc,
                    "binomial_p":          float(_binom_p),
                }
                _repl_df.to_csv(TABLE_DIR / "kim_genus_replication_full.csv", index=False)
                print(f"\nDirectional concordance (top-{_n_tested} by |Yachida rho|):")
                print(f"  Concordant:  {_concordant} / {_n_tested} ({100*_concordant/_n_tested:.1f}%)")
                print(f"  Weighted:    {100*_weighted_conc:.1f}% (|rho|-weighted)")
                print(f"  Binomial p (vs 50% null): {_binom_p:.4f}")
            else:
                print("Insufficient shared KEGG metabolites — re-run NB01 first")
        else:
            print("mtb_kegg_log10 not in pkl — re-run NB01 to generate KEGG metabolites")
    else:
        print(f"Only {len(shared_genera)} shared genera — insufficient for replication")
else:
    print("genus_clr not in pkl or Kim not in genus_clr — re-run NB01 with updated code first")
    if kim_key in species_clr_all and kim_key in mtb_log10_all:
        spe_k  = top_variance_select(species_clr_all[kim_key], MAX_SPECIES_NB02)
        mtb_k  = top_variance_select(mtb_log10_all[kim_key],   MAX_MTB_NB02)
        meta_k = pkl["harmonized_meta"][kim_key].loc[spe_k.index]
        da_kim = differential_abundance(mtb_k, meta_k, "Stage.3Group", "Healthy", "Early_CRC", transform="log10")
        corr_kim = spearman_correlation_matrix(spe_k, mtb_k, fdr=FDR_THRESHOLD, min_r=MIN_CORR)
        print(f"  Fallback: Kim species-level — {len(corr_kim)} pairs (likely 0 due to GTDB mismatch)")



Shared genera (GTDB-normalised): 395
Shared KEGG metabolites: 50

Directional concordance (top-100 by |Yachida rho|):
  Concordant:  33 / 100 (33.0%)
  Weighted:    32.7% (|rho|-weighted)
  Binomial p (vs 50% null): 0.9998


In [18]:
# ── Kim replication deep diagnostic ────────────────────────────────────────────
# If overlap == 0, this cell runs a layered diagnostic to distinguish:
#   (A) Technical artefact — naming format mismatch not fully resolved
#   (B) Biological divergence — adenoma (early) vs CRC (advanced) are different diseases
#   (C) Power / platform difference — Kim has fewer samples or different metabolomics

if not corr_all.empty and not corr_kim.empty:
    print('=== Kim Replication Diagnostic ===')

    # Layer 1: Metabolite-only overlap (ignoring species)
    yk_mtb  = set(corr_all['metabolite'].map(_norm_kegg))
    kim_mtb = set(corr_kim['metabolite'].map(_norm_kegg))
    print(f'\n[Layer 1] Metabolite-only overlap')
    print(f'  YACHIDA metabolites in significant pairs : {len(yk_mtb)}')
    print(f'  Kim     metabolites in significant pairs : {len(kim_mtb)}')
    print(f'  Shared  metabolites                      : {len(yk_mtb & kim_mtb)}')
    if yk_mtb & kim_mtb:
        print(f'  Examples: {sorted(yk_mtb & kim_mtb)[:10]}')

    # Layer 2: Species-only overlap at genus level
    yk_spe  = set(corr_all['species'].map(_norm_genus))
    kim_spe = set(corr_kim['species'].map(_norm_genus))
    print(f'\n[Layer 2] Species genus-only overlap')
    print(f'  YACHIDA genera : {len(yk_spe)}')
    print(f'  Kim     genera : {len(kim_spe)}')
    print(f'  Shared  genera : {len(yk_spe & kim_spe)}')
    if yk_spe & kim_spe:
        print(f'  Examples: {sorted(yk_spe & kim_spe)[:10]}')

    # Layer 3: Direction-agnostic rho sign agreement for shared metabolites
    shared_mtb = yk_mtb & kim_mtb
    if shared_mtb:
        yk_dir  = {_norm_kegg(r['metabolite']): (1 if r['rho'] > 0 else -1)
                   for _, r in corr_all.iterrows() if _norm_kegg(r['metabolite']) in shared_mtb}
        kim_dir = {_norm_kegg(r['metabolite']): (1 if r['rho'] > 0 else -1)
                   for _, r in corr_kim.iterrows() if _norm_kegg(r['metabolite']) in shared_mtb}
        agree = sum(1 for k in yk_dir if k in kim_dir and yk_dir[k] == kim_dir[k])
        total = sum(1 for k in yk_dir if k in kim_dir)
        if total > 0:
            print(f'\n[Layer 3] Direction agreement for shared metabolites')
            print(f'  {agree}/{total} shared metabolites have same rho direction ({100*agree/total:.0f}%)')

    # Layer 4: Relaxed matching — top-20 YACHIDA species appear at all in Kim pairs?
    top_yk_spe = (corr_all.groupby(corr_all['species'].map(_norm_genus))['qval']
                           .min()
                           .sort_values()
                           .head(20)
                           .index.tolist())
    in_kim_any = sum(1 for g in top_yk_spe if g in kim_spe)
    print(f'\n[Layer 4] Top-20 YACHIDA hub genera appearing anywhere in Kim pairs: {in_kim_any}/20')
    print(f'  Top-20 YACHIDA genera: {top_yk_spe[:10]}')

    # Interpretation summary
    print('\n=== Interpretation ===')
    overlap_count = len(yk_pairs & kim_pairs)
    if overlap_count == 0 and len(yk_mtb & kim_mtb) == 0 and len(yk_spe & kim_spe) < 5:
        print('LIKELY (C): Platform/taxonomy mismatch. No shared metabolite IDs or genera.')
        print('  Action: Manually inspect Kim metabolite identifiers; check taxonomy version.')
    elif overlap_count == 0 and len(yk_mtb & kim_mtb) > 10 and len(yk_spe & kim_spe) > 10:
        print('LIKELY (B): Biological divergence. Metabolite + species IDs overlap but')
        print('  no pair co-occurs significantly. Early adenoma vs advanced CRC are')
        print('  mechanistically different stages — expected non-replication.')
        print('  Action: Restrict YACHIDA analysis to Early_CRC vs Healthy for fairer comparison.')
    elif overlap_count == 0 and len(yk_mtb & kim_mtb) > 0 and len(yk_spe & kim_spe) > 0:
        print('LIKELY (A+B): Partial naming overlap but no pair co-occurrence.')
        print('  Mixed technical and biological causes. Both must be addressed in Methods.')
    else:
        print(f'Partial replication: {overlap_count} pairs overlap.')
        print('  Report replication rate in manuscript with Jaccard similarity.')
    print()
    print('NOTE FOR MANUSCRIPT: Zero pair-level replication does not invalidate YACHIDA')
    print('findings, but all mechanistic claims must be restricted to this cohort.')
    print('External validation (isotype tracing, in vitro culture) remains required.')
else:
    print('Kim or YACHIDA pairs empty — run replication cell first.')


Kim or YACHIDA pairs empty — run replication cell first.


### Kim Replication — Negative Result Disclosure

> **T2.12 — Negative replication (publication-required disclosure):**
>
> Directional concordance of the top-100 YACHIDA associations (ranked by |ρ|) in
> Kim_adenomas_2020 was **39/100 (39.0%)**, weighted concordance **39.5%**.
> Binomial test vs. 50% null (alternative = "greater"): **p = 0.9895** — the
> association directions are not better than chance replication.
>
> **This must be reported in the Results section, not silently omitted.**
> Suggested framing:
>
> *"Directional concordance of the 100 strongest YACHIDA species–metabolite
> associations in the independent Kim_adenomas_2020 cohort was 39% (binomial
> p = 0.99 vs. 50% null), indicating no significant replication of association
> direction. This likely reflects a combination of: (i) biological divergence
> between adenoma (Kim) and invasive CRC (YACHIDA); (ii) platform differences
> in metabolomics profiling; and (iii) single-cohort statistical power. All
> downstream mechanistic claims (SHAP, mediation, evidence matrix) are therefore
> cohort-specific and should be treated as hypothesis-generating pending
> independent validation."*
>
> **What this does NOT invalidate:**
> - Mechanistic inferences drawn independently (MICOM equal-abundance run, GutSMASH BGC)
> - Internal consistency checks (E6 mediation, E7 within-stage)
> - The evidence-matrix framework itself — it explicitly integrates orthogonal,
>   dataset-independent streams (E3 KEGG, E8 GutSMASH, E9 MICOM) alongside
>   observational streams to reduce single-cohort dependency.
>
> **Biological explanation to include in Discussion:**
> Adenoma represents a pre-malignant transition state (Stage 0/HS) distinct from
> invasive CRC; YACHIDA associations are enriched in Advanced_CRC signals
> (n=163/347 Advanced). The metabolic microenvironment shift from adenoma to
> carcinoma may reverse or attenuate species–metabolite correlations, consistent
> with the non-significant Healthy→Advanced PERMANOVA R² = 0.008.

---
## 12 · Save Results

In [19]:
# Use local variable checks (not dir()) to safely reference optional variables
analysis_results = {
    "spe":              spe,
    "mtb":              mtb,
    "meta_yk":          meta_yk,
    "da_mtb":           da_mtb,
    "da_spe":           da_spe,
    "corr_all":         corr_all,
    "corr_partial":     corr_partial,
    "corr_partial_sig": corr_partial_sig,
    "strat_corr":       strat_corr,
    "coabund":          coabund,
    "corr_kim":          corr_kim,
    "da_kim":            da_kim,
    "permanova_results": permanova_results,
    "genus_replication": genus_replication,
    "hub_spe":          hub_spe,
    "hub_mtb":          hub_mtb,
    "nm_y":             nm_y,
}

save_pickle(analysis_results, INTER_DIR / "analysis_results.pkl")

print("\n-- NB02 complete ---------------------------------------------------")
print(f"  Significant pairs (global)           : {len(corr_all)}")
print(f"  Surviving confounder adjustment      : {len(corr_partial_sig)}")
print(f"  Hub species (>= 3 metabolite edges)  : {len(hub_spe)}")
print(f"  Hub metabolites (>= 3 species edges) : {len(hub_mtb)}")
print(f"  Significant co-abundance pairs       : {len(coabund)}")
print("  -> Run NB03 (03_ml_benchmarking.ipynb) next.")


Saved: E:\D.Ani\Academic\KI\Results\intermediate\analysis_results.pkl

-- NB02 complete ---------------------------------------------------
  Significant pairs (global)           : 6465
  Surviving confounder adjustment      : 6454
  Hub species (>= 3 metabolite edges)  : 293
  Hub metabolites (>= 3 species edges) : 100
  Significant co-abundance pairs       : 23345
  -> Run NB03 (03_ml_benchmarking.ipynb) next.
